# 5. Feature selection: pairwise alignment experiment <a id="5"></a>


## Table of contents

- [5.1 Setup & pairwise run](#setup)
- [5.2 Comparison: runtime](#cmp-runtime)
- [5.3 Comparison: alignment coverage](#cmp-coverage)
- [5.4 Comparison: conservation overlap](#cmp-overlap)


## Backend map

How this notebook connects to `workflow/` modules:

```mermaid
flowchart LR
  nb["08c-FeatureSelection"]
  m0["workflow.pairwise_alignment_experiment"]
  nb --> m0
```


This notebook compares **pairwise** structural alignment of every structure in `Results/activation_segments/misaligned_filter/` to the BRAF reference `6UAN_chainD.pdb`, using:

| Method | API | Inputs per call |
|---|---|---|
| FoldMason | `AlignmentFoldMason.process_foldmason_alignment` | template + one PDB |
| MUSTANG | `AlignmentMustang.process_mustang_alignment` | template + one PDB |

Comparison panels match the multi-MSA experiment in `08b`: **runtime**, **coverage** (mean ± SEM across pairs), and **conservation overlap** at ≥70%.

This can be slow on the full dataset; progress is written to logs under `Results/Experiments/pairwise_alignment/`.

The multi-seed multi-MSA experiment remains in `08b`; production FoldMason conservation remains in `08a`.


To get started, let's load some packages!


In [ ]:
import pandas as pd
from IPython.display import display, HTML

from workflow.pairwise_alignment_experiment import (
    run_pairwise_comparison,
    plot_runtime_comparison,
    plot_coverage_comparison,
    plot_conservation_overlap,
)


## 5.1 Setup & pairwise run <a id="setup"></a>

For each PDB in `Results/activation_segments/misaligned_filter/`, both methods align **template + that PDB** only. Outputs are written under `Results/Experiments/pairwise_alignment/{foldmason,mustang}/`.


In [ ]:
PDB_DIR = "Results/activation_segments/misaligned_filter/"
TEMPLATE_PDB = "6UAN_chainD.pdb"
EXPERIMENT_DIR = "Results/Experiments/pairwise_alignment"
MUSTANG_BIN = "/home/marmatt/Downloads/MUSTANG_v3.2.4/bin/mustang-3.2.4"

results = run_pairwise_comparison(
    pdb_dir=PDB_DIR,
    template_pdb=TEMPLATE_PDB,
    experiment_dir=EXPERIMENT_DIR,
    mustang_bin=MUSTANG_BIN,
    conservation_threshold=0.70,
    show_conservation_plots=False,
)

summary = results["summary"]
coverage_df = results["coverage_df"]
display(pd.DataFrame([summary]))
display(coverage_df.head(20))
print(f"(showing first 20 of {len(coverage_df)} per-pair coverage rows)")


## 5.2 Comparison: runtime <a id="cmp-runtime"></a>

Total wall-clock time for the full pairwise loop (FoldMason vs MUSTANG).


In [ ]:
runtime_summary = plot_runtime_comparison(
    summary,
    experiment_dir=EXPERIMENT_DIR,
    show=True,
)
display(runtime_summary)


## 5.3 Comparison: alignment coverage <a id="cmp-coverage"></a>

Per successful pairwise alignment we record alignment length, fully aligned columns, and mean gap fraction. Bars show **mean ± SEM across pairs**.


In [ ]:
coverage_summary = plot_coverage_comparison(
    coverage_df,
    experiment_dir=EXPERIMENT_DIR,
    show=True,
)
display(coverage_summary)


## 5.4 Comparison: conservation overlap <a id="cmp-overlap"></a>

Residues conserved at ≥70% in the pairwise-to-reference conservation analysis for each method; overlap of those sets (FoldMason only / both / MUSTANG only) and Jaccard similarity.


In [ ]:
overlap_summary = plot_conservation_overlap(
    summary,
    experiment_dir=EXPERIMENT_DIR,
    show=True,
)
display(overlap_summary)
